# Smart Grid Silver Layer (Capstone 09, Day 2)

Deterministic, idempotent rebuild of `capstone9_smartmeter.smart_grid_silver` from immutable Bronze. Every rule has an ID, severity and action in `silver_dq_rule_catalog`. **ERROR** rules quarantine the row (`quarantine_*` tables, with `_dq_reason`, `_dq_rule_ids` and the raw Bronze record). **WARN** rules keep the row and flag it in `_dq_warnings`. Duplicates and replays are dropped and audited in `silver_duplicate_audit`. Domain outputs are `silver_meter_expected_intervals`, `silver_meter_interval_gaps` and `silver_meter_daily_energy`, with controls in `silver_reconciliation`. Run all cells top to bottom; the snapshot cell must run before the rebuild so the before/after comparison is meaningful.

In [0]:

tables = [
    "bronze_smart_meter_readings",
    "bronze_grid_outage_events",
    "bronze_meter_recovery_events",
    "bronze_meter_master",
    "bronze_substation_master",
    "bronze_tariff_lookup"
]

for table in tables:
    full_name = f"capstone9_smartmeter.smart_grid_bronze.{table}"

    print("\n" + "=" * 70)
    print("TABLE:", table)

    df = spark.table(full_name)

    print("Record count:", df.count())
    df.printSchema()
    display(df.limit(5))


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# UTC for every date/timestamp transformation (source timestamps end in 'Z'):
# reading_date and expected-interval dates = UTC calendar day, identical to Gold (04)
spark.conf.set("spark.sql.session.timeZone", "UTC")

# --------------------------------------------------
# 1. CATALOG AND SCHEMA
# --------------------------------------------------

catalog = "capstone9_smartmeter"
bronze_schema = "smart_grid_bronze"
silver_schema = "smart_grid_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

def fq(table):
    return f"{catalog}.{silver_schema}.{table}"

# --------------------------------------------------
# 1b. RUN METADATA AND BUSINESS-RULE THRESHOLDS
# (documented in the DQ rule catalogue table)
# --------------------------------------------------
import uuid
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]

INTERVAL_MINUTES = 5                  # smart meter interval length
ENERGY_SENTINELS = [-999.0, 999999.99] # placeholder values found in the feed
ENERGY_MAX_KWH = 50.0                 # max plausible kWh per 5-min interval (observed p99 = 11.9)
VOLTAGE_RANGE = (180.0, 280.0)        # V
CURRENT_RANGE = (0.0, 200.0)          # A
POWER_FACTOR_RANGE = (0.0, 1.0)
SIGNAL_DBM_RANGE = (-120.0, 0.0)      # signal_strength is reported in dBm (negative)
LATE_READING_MINUTES = 60             # received_ts - interval_end_ts
LATE_OUTAGE_REPORT_MINUTES = 60       # reported_ts - outage_start_ts
LATE_RECOVERY_MINUTES = 60            # received_ts - event_ts

print("Silver run id:", run_id)

In [0]:
# --------------------------------------------------
# 1c. SNAPSHOT OF THE CURRENT SILVER LAYER ("BEFORE" STATE)
# Captured before this run rebuilds Silver, used by the comparison cell.
# --------------------------------------------------
SILVER_TABLES = [
    "silver_substation_master",
    "silver_tariff_lookup",
    "silver_meter_master",
    "silver_smart_meter_readings",
    "silver_grid_outage_events",
    "silver_meter_recovery_events",
]

def table_exists(t):
    return spark.catalog.tableExists(fq(t))

pre_counts, pre_reasons = [], []
for t in SILVER_TABLES:
    q = t.replace("silver_", "quarantine_")
    v = spark.table(fq(t)).count() if table_exists(t) else None
    qc = spark.table(fq(q)).count() if table_exists(q) else None
    pre_counts.append((t, v, qc))
    if table_exists(q) and "_dq_reason" in spark.table(fq(q)).columns:
        for r in spark.table(fq(q)).groupBy("_dq_reason").count().collect():
            pre_reasons.append((t, r["_dq_reason"], r["count"]))

pre_metrics = {}
if table_exists("silver_smart_meter_readings"):
    m = spark.table(fq("silver_smart_meter_readings")).agg(
        F.round(F.sum("energy_kwh"), 2).alias("kwh"),
        F.max("energy_kwh").alias("max_kwh"),
        F.sum((F.col("energy_kwh") > ENERGY_MAX_KWH).cast("int")).alias("extreme"),
    ).first()
    pre_metrics.update({
        "Total kWh in silver readings": m["kwh"],
        "Max interval kWh in silver readings": m["max_kwh"],
        f"Silver readings with energy_kwh > {ENERGY_MAX_KWH}": m["extreme"],
    })
if table_exists("silver_meter_master"):
    pre_metrics["Meter tariff_code values with hidden CR/LF"] = (
        spark.table(fq("silver_meter_master"))
        .filter(F.col("tariff_code").rlike("[\\r\\n]")).count()
    )
if table_exists("silver_meter_recovery_events"):
    pre_metrics["Recovery events with acknowledged_flag populated"] = (
        spark.table(fq("silver_meter_recovery_events"))
        .filter(F.col("acknowledged_flag").isNotNull()).count()
    )

pre_counts_df = spark.createDataFrame(
    pre_counts, "table string, valid_before long, quarantined_before long"
)
display(pre_counts_df)
print(pre_metrics)

In [0]:
# --------------------------------------------------
# 2. HELPER FUNCTIONS - DQ FRAMEWORK
# Pure transformations (unit-tested at the end of the notebook) + publish (I/O)
# --------------------------------------------------
from functools import reduce

# Leading/trailing spaces AND control characters (e.g. the trailing \r left by
# CRLF Azure SQL extracts on the last column), plus non-breaking space
CTRL_WS = r"^[\x00-\x20\xA0]+|[\x00-\x20\xA0]+$"
DQ_STRUCT = "array<struct<rule_id:string,reason:string>>"

RULE_CATALOG = {}      # rule_id -> metadata (written to silver_dq_rule_catalog)
# Keyed by silver table so re-running a cell replaces (not duplicates) its entries
RUN_METRICS = {}       # one dict per published entity
REJECTED_FRAMES = {}   # unified rejected-record audit
DUPLICATE_FRAMES = {}  # duplicate / replay audit


def rule(entity, rule_id, severity, columns, reason, cond):
    """Register a rule. cond = PASS condition. ERROR -> quarantine, WARN -> flag and keep."""
    RULE_CATALOG[rule_id] = {
        "entity": entity, "rule_id": rule_id, "severity": severity,
        "columns": ", ".join(columns), "reason": reason,
        "action": "QUARANTINE" if severity == "ERROR" else "FLAG_AND_KEEP",
    }
    return {"rule_id": rule_id, "severity": severity, "reason": reason, "cond": cond}


def not_null_rules(entity, prefix, cols, start=1):
    return [
        rule(entity, f"{prefix}-{start + i:03d}", "ERROR", [c],
             f"Null validation failed: {c}", F.col(c).isNotNull())
        for i, c in enumerate(cols)
    ]


def type_rules(entity, prefix, specs, start):
    return [
        rule(entity, f"{prefix}-{start + i:03d}", "ERROR", [c],
             f"Type validation failed: {c} is not a valid {t}",
             F.col(f"_orig_{c}").isNull() | F.col(c).isNotNull())
        for i, (c, t) in enumerate(specs.items())
    ]


def rescued_rule(entity, rule_id, handled=()):
    """Schema contract: _rescued_data may only contain keys that Silver recovers."""
    allowed = F.array(*[F.lit(k) for k in ("_file_path",) + tuple(handled)])
    unhandled = F.array_except(F.expr("json_object_keys(_rescued_data)"), allowed)
    return rule(entity, rule_id, "ERROR", ["_rescued_data"],
                "Schema validation failed: unparsed values in _rescued_data",
                F.col("_rescued_data").isNull() | (F.size(unhandled) == 0))


# ---------------- cleaning / typing ----------------
def clean_strings(df):
    """Trim whitespace + control chars on business string columns; blank -> NULL."""
    out = []
    for c, t in df.dtypes:
        if t == "string" and not c.startswith("_"):
            v = F.regexp_replace(F.col(c), CTRL_WS, "")
            out.append(F.when(v == "", F.lit(None).cast("string")).otherwise(v).alias(c))
        else:
            out.append(F.col(c))
    return df.select(*out)


def add_lineage(df):
    """Keep the untouched Bronze record as JSON + unify lineage column names."""
    biz = [c for c in df.columns if not c.startswith("_")]
    df = df.withColumn("_raw_record", F.to_json(F.struct(*biz)))
    if "_source_file" not in df.columns:
        df = (df.withColumnRenamed("_source", "_source_file") if "_source" in df.columns
              else df.withColumn("_source_file", F.lit(None).cast("string")))
    if "_rescued_data" not in df.columns:
        df = df.withColumn("_rescued_data", F.lit(None).cast("string"))
    return df


def read_bronze(table):
    return clean_strings(add_lineage(spark.table(f"{catalog}.{bronze_schema}.{table}")))


def cast_columns(df, specs):
    """try_cast each column; original kept in _orig_<col> for the type rule."""
    for c, t in specs.items():
        df = (df.withColumn(f"_orig_{c}", F.col(c))
                .withColumn(c, F.expr(f"try_cast(`{c}` AS {t})")))
    return df


def flag_expr(col):
    v = F.lower(col)
    return (F.when(v.isin("true", "1", "yes", "y", "active"), True)
             .when(v.isin("false", "0", "no", "n", "inactive"), False))


def standardize_flag(df, column):
    return df.withColumn(f"_orig_{column}", F.col(column).cast("string")) \
             .withColumn(column, flag_expr(F.col(column).cast("string")))


def add_ref_flag(df, ref_df, on, flag):
    """Left-join a distinct key set and add a boolean 'key exists in parent' flag."""
    keys = ref_df.select(*on).dropna().dropDuplicates().withColumn(flag, F.lit(True))
    return (df.join(F.broadcast(keys), on, "left")
              .withColumn(flag, F.coalesce(F.col(flag), F.lit(False))))


# ---------------- deduplication / arrival ----------------
def dedupe_with_audit(df, keys, order_cols, entity, rule_id, label="DUPLICATE"):
    """Deterministic dedup. Keeps rank 1 per key (order_cols, then payload hash as final
    tie-breaker). Returns (kept, dropped); dropped rows are typed EXACT_/CONFLICTING_<label>.
    Rows with a NULL key bypass dedup and are caught by the null rules."""
    rule(entity, rule_id, "INFO", keys,
         f"Duplicate validation failed: {label.lower()} for key ({', '.join(keys)})", F.lit(True))
    RULE_CATALOG[rule_id]["action"] = "DROP_AND_AUDIT"
    payload = [c for c in df.columns if not c.startswith("_")]
    null_key = reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys])
    bypass, cand = df.filter(null_key), df.filter(~null_key)

    w = Window.partitionBy(*keys).orderBy(*order_cols, F.col("_payload_hash"))
    cand = (cand.withColumn("_payload_hash", F.sha2(F.to_json(F.struct(*payload)), 256))
                .withColumn("_rn", F.row_number().over(w))
                .withColumn("_kept_hash", F.first("_payload_hash").over(w))
                .withColumn("_n", F.count(F.lit(1)).over(Window.partitionBy(*keys))))
    prev = F.coalesce(F.col("_duplicate_count"), F.lit(0)) if "_duplicate_count" in df.columns else F.lit(0)
    kept = (cand.filter("_rn = 1")
                .withColumn("_duplicate_count", (prev + F.col("_n") - 1).cast("int"))
                .drop("_rn", "_kept_hash", "_n", "_payload_hash"))
    if "_duplicate_count" not in bypass.columns:
        bypass = bypass.withColumn("_duplicate_count", F.lit(0))
    kept = kept.unionByName(bypass)

    exact = F.col("_payload_hash") == F.col("_kept_hash")
    dropped = (cand.filter("_rn > 1")
        .withColumn("_duplicate_type",
                    F.when(exact, F.lit(f"EXACT_{label}")).otherwise(F.lit(f"CONFLICTING_{label}")))
        .withColumn("_dq_rule_ids", F.array(F.lit(rule_id)))
        .withColumn("_dq_reason", F.concat(
            F.lit("Duplicate validation failed: "),
            F.when(exact, F.lit(f"exact {label.lower()}"))
             .otherwise(F.lit(f"conflicting {label.lower()} superseded by later version")),
            F.lit(f" for key ({', '.join(keys)})")))
        .drop("_rn", "_kept_hash", "_n", "_payload_hash"))
    return kept, dropped


def flag_arrival(df, entity_keys, event_col, received_col, late_minutes):
    """_is_late_arrival: received too long after event time.
    _is_out_of_order: an event that arrived after a LATER event of the same entity."""
    lag_min = (F.unix_timestamp(received_col) - F.unix_timestamp(event_col)) / 60
    w = (Window.partitionBy(*entity_keys)
               .orderBy(F.col(received_col), F.col(event_col))
               .rowsBetween(Window.unboundedPreceding, -1))
    return (df.withColumn("_arrival_lag_minutes", lag_min)
              .withColumn("_is_late_arrival", F.coalesce(lag_min > late_minutes, F.lit(False)))
              .withColumn("_is_out_of_order",
                          F.coalesce(F.col(event_col) < F.max(event_col).over(w), F.lit(False))))


# ---------------- rule evaluation ----------------
def evaluate_rules(df, rules):
    """Adds _dq_errors / _dq_warns arrays of (rule_id, reason). A NULL rule result
    (indeterminate) passes - missing values are owned by the explicit null rules."""
    def collect(sev):
        items = [
            F.when(~F.coalesce(r["cond"], F.lit(True)),
                   F.struct(F.lit(r["rule_id"]).alias("rule_id"), F.lit(r["reason"]).alias("reason")))
            for r in rules if r["severity"] == sev
        ]
        if not items:
            return F.array().cast(DQ_STRUCT)
        return F.filter(F.array(*items), lambda x: x.isNotNull())
    return df.withColumn("_dq_errors", collect("ERROR")).withColumn("_dq_warns", collect("WARN"))


def split_valid_rejected(df):
    reasons = lambda c: F.transform(c, lambda x: x["reason"])
    ids = lambda c: F.transform(c, lambda x: x["rule_id"])
    tech = [c for c in df.columns if c.startswith(("_orig_", "_ref_"))]
    base = (df.withColumn("_dq_warnings", reasons("_dq_warns"))
              .withColumn("_dq_warning_rule_ids", ids("_dq_warns")))
    valid = (base.filter(F.size("_dq_errors") == 0)
                 .drop("_dq_errors", "_dq_warns", "_raw_record", "_rescued_data", *tech))
    rejected = (base.filter(F.size("_dq_errors") > 0)
                    .withColumn("_dq_rule_ids", ids("_dq_errors"))
                    .withColumn("_dq_reason", F.array_join(reasons("_dq_errors"), "; "))
                    .drop("_dq_errors", "_dq_warns", *tech))
    return valid, rejected


# ---------------- publish (I/O) ----------------
def _overwrite(df, table):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(fq(table)))


def publish(df, dropped, silver_table, entity, key_cols, rules, bronze_rows):
    """Full deterministic rebuild from immutable Bronze -> idempotent on replay."""
    valid, rejected = split_valid_rejected(evaluate_rules(df, rules))
    stamp = lambda d: (d.withColumn("_silver_run_id", F.lit(run_id))
                        .withColumn("_silver_processed_at", F.current_timestamp()))
    quarantine_table = silver_table.replace("silver_", "quarantine_")
    _overwrite(stamp(valid), silver_table)
    _overwrite(stamp(rejected), quarantine_table)

    bkey = F.concat_ws("|", *[F.col(k).cast("string") for k in key_cols])
    audit_cols = lambda d, stage: d.select(
        F.lit(entity).alias("entity"), F.lit(silver_table).alias("target_table"),
        bkey.alias("business_key"), F.lit(stage).alias("dq_stage"),
        F.col("_dq_rule_ids").alias("rule_ids"), F.col("_dq_reason").alias("dq_reason"),
        F.col("_source_file").alias("source_file"), F.col("_ingested_at").alias("bronze_ingested_at"),
        F.col("_raw_record").alias("raw_record"), F.lit(run_id).alias("silver_run_id"))
    REJECTED_FRAMES[silver_table] = [
        audit_cols(spark.table(fq(quarantine_table)), "VALIDATION"),
        audit_cols(dropped, "DEDUPLICATION"),
    ]
    DUPLICATE_FRAMES[silver_table] = (dropped.select(
        F.lit(entity).alias("entity"), bkey.alias("business_key"),
        F.col("_duplicate_type").alias("duplicate_type"),
        F.col("_dq_reason").alias("dq_reason"),
        F.col("_source_file").alias("source_file"),
        F.col("_ingested_at").alias("bronze_ingested_at"),
        F.col("_raw_record").alias("raw_record"),
        F.lit(run_id).alias("silver_run_id")))

    out = spark.table(fq(silver_table))
    v, q, d = out.count(), spark.table(fq(quarantine_table)).count(), dropped.count()
    w = out.filter(F.size("_dq_warnings") > 0).count()
    RUN_METRICS[silver_table] = {
        "entity": entity, "silver_table": silver_table, "bronze_rows": bronze_rows,
        "silver_valid": v, "quarantined": q, "duplicates_removed": d,
        "valid_with_warnings": w, "balanced": bronze_rows == v + q + d,
    }
    print(f"{silver_table}: bronze={bronze_rows} -> valid={v}, quarantined={q}, "
          f"duplicates_removed={d}, valid_with_warnings={w}, balanced={bronze_rows == v + q + d}")
    return out

In [0]:
# --------------------------------------------------
# 3. SUBSTATION MASTER (Azure SQL seed extract, business key substation_id)
# --------------------------------------------------
E_SUB = "substation_master"
SUB_TYPES = {"capacity_mva": "DOUBLE", "voltage_level_kv": "DOUBLE",
             "commissioned_date": "DATE", "latitude": "DOUBLE", "longitude": "DOUBLE"}
MASTER_ORDER = [F.col("_ingested_at").desc_nulls_last(), F.col("_source_file").desc_nulls_last()]

bronze_sub = read_bronze("bronze_substation_master")
sub = standardize_flag(cast_columns(bronze_sub, SUB_TYPES), "active_flag")
sub, sub_dups = dedupe_with_audit(sub, ["substation_id"], MASTER_ORDER, E_SUB, "SUB-DUP-001")

c = F.col
SUB_RULES = (
    not_null_rules(E_SUB, "SUB", ["substation_id", "substation_name"])          # 001-002
    + type_rules(E_SUB, "SUB", SUB_TYPES, 3)                                     # 003-007
    + [
        rule(E_SUB, "SUB-008", "ERROR", ["active_flag"],
             "Domain validation failed: active_flag must be Y/N",
             c("_orig_active_flag").isNull() | c("active_flag").isNotNull()),
        rule(E_SUB, "SUB-009", "ERROR", ["capacity_mva"],
             "Range validation failed: capacity_mva must be > 0",
             c("capacity_mva").isNull() | (c("capacity_mva") > 0)),
        rule(E_SUB, "SUB-010", "ERROR", ["voltage_level_kv"],
             "Range validation failed: voltage_level_kv must be > 0",
             c("voltage_level_kv").isNull() | (c("voltage_level_kv") > 0)),
        rule(E_SUB, "SUB-011", "ERROR", ["latitude"],
             "Range validation failed: latitude must be between -90 and 90",
             c("latitude").isNull() | c("latitude").between(-90, 90)),
        rule(E_SUB, "SUB-012", "ERROR", ["longitude"],
             "Range validation failed: longitude must be between -180 and 180",
             c("longitude").isNull() | c("longitude").between(-180, 180)),
        rule(E_SUB, "SUB-013", "WARN", ["commissioned_date"],
             "Date validation warning: commissioned_date is in the future",
             c("commissioned_date").isNull() | (c("commissioned_date") <= F.current_date())),
    ]
)

silver_substation = publish(sub, sub_dups, "silver_substation_master", E_SUB,
                            ["substation_id"], SUB_RULES, bronze_sub.count())

In [0]:
# --------------------------------------------------
# 4. TARIFF LOOKUP (Azure SQL seed extract, business key tariff_code)
# --------------------------------------------------
E_TRF = "tariff_lookup"
TRF_TYPES = {"rate_per_kwh": "DOUBLE", "effective_date": "DATE",
             "expiry_date": "DATE", "peak_multiplier": "DOUBLE"}

bronze_trf = read_bronze("bronze_tariff_lookup")
trf = standardize_flag(cast_columns(bronze_trf, TRF_TYPES), "active_flag")
# Handbook grain: one record per tariff_code; latest effective version wins
trf, trf_dups = dedupe_with_audit(
    trf, ["tariff_code"],
    [F.col("effective_date").desc_nulls_last()] + MASTER_ORDER, E_TRF, "TRF-DUP-001")

TRF_RULES = (
    not_null_rules(E_TRF, "TRF", ["tariff_code", "customer_class", "rate_per_kwh", "effective_date"])  # 001-004
    + type_rules(E_TRF, "TRF", TRF_TYPES, 5)                                                         # 005-008
    + [
        rule(E_TRF, "TRF-009", "ERROR", ["active_flag"],
             "Domain validation failed: active_flag must be Y/N",
             c("_orig_active_flag").isNull() | c("active_flag").isNotNull()),
        rule(E_TRF, "TRF-010", "ERROR", ["customer_class"],
             "Domain validation failed: customer_class must be RESIDENTIAL, COMMERCIAL or INDUSTRIAL",
             c("customer_class").isNull() | c("customer_class").isin("RESIDENTIAL", "COMMERCIAL", "INDUSTRIAL")),
        rule(E_TRF, "TRF-011", "ERROR", ["rate_per_kwh"],
             "Range validation failed: rate_per_kwh must be >= 0",
             c("rate_per_kwh").isNull() | (c("rate_per_kwh") >= 0)),
        rule(E_TRF, "TRF-012", "ERROR", ["peak_multiplier"],
             "Range validation failed: peak_multiplier must be > 0",
             c("peak_multiplier").isNull() | (c("peak_multiplier") > 0)),
        rule(E_TRF, "TRF-013", "ERROR", ["effective_date", "expiry_date"],
             "Date validation failed: expiry_date before effective_date",
             c("expiry_date").isNull() | (c("expiry_date") >= c("effective_date"))),
        rule(E_TRF, "TRF-014", "ERROR", ["currency"],
             "Domain validation failed: currency must be a 3-letter ISO code",
             c("currency").isNull() | c("currency").rlike("^[A-Z]{3}$")),
    ]
)

silver_tariff = publish(trf, trf_dups, "silver_tariff_lookup", E_TRF,
                        ["tariff_code"], TRF_RULES, bronze_trf.count())

In [0]:
# --------------------------------------------------
# 5. METER MASTER (Azure SQL seed extract, business key meter_id)
# Fix: the CRLF extract left a hidden \r on tariff_code (last column) -> cleaned by read_bronze.
# --------------------------------------------------
E_MTR = "meter_master"
MTR_TYPES = {"installation_date": "DATE", "rated_voltage": "DOUBLE"}

bronze_mtr = read_bronze("bronze_meter_master")
mtr = cast_columns(bronze_mtr, MTR_TYPES)
mtr, mtr_dups = dedupe_with_audit(mtr, ["meter_id"], MASTER_ORDER, E_MTR, "MTR-DUP-001")
mtr = add_ref_flag(mtr, silver_substation, ["substation_id"], "_ref_substation_ok")
mtr = add_ref_flag(mtr, silver_tariff, ["tariff_code"], "_ref_tariff_ok")
# Keep the tariff resolution result as a business column for Gold
mtr = mtr.withColumn("is_tariff_resolved", F.col("_ref_tariff_ok"))

MTR_RULES = (
    not_null_rules(E_MTR, "MTR", ["meter_id", "substation_id", "meter_status", "customer_class"])  # 001-004
    + type_rules(E_MTR, "MTR", MTR_TYPES, 5)                                                        # 005-006
    + [
        rule(E_MTR, "MTR-007", "ERROR", ["meter_status"],
             "Domain validation failed: meter_status must be ACTIVE, INACTIVE or PENDING",
             c("meter_status").isNull() | c("meter_status").isin("ACTIVE", "INACTIVE", "PENDING")),
        rule(E_MTR, "MTR-008", "ERROR", ["customer_class"],
             "Domain validation failed: customer_class must be RESIDENTIAL, COMMERCIAL or INDUSTRIAL",
             c("customer_class").isNull() | c("customer_class").isin("RESIDENTIAL", "COMMERCIAL", "INDUSTRIAL")),
        rule(E_MTR, "MTR-009", "ERROR", ["rated_voltage"],
             "Range validation failed: rated_voltage must be between 1 and 1000",
             c("rated_voltage").isNull() | c("rated_voltage").between(1, 1000)),
        rule(E_MTR, "MTR-010", "ERROR", ["substation_id"],
             "Referential validation failed: substation_id not found in silver_substation_master",
             c("substation_id").isNull() | c("_ref_substation_ok")),
        rule(E_MTR, "MTR-011", "WARN", ["tariff_code"],
             "Referential validation warning: tariff_code not found in silver_tariff_lookup",
             c("tariff_code").isNull() | c("_ref_tariff_ok")),
        rule(E_MTR, "MTR-012", "WARN", ["installation_date"],
             "Date validation warning: installation_date is in the future",
             c("installation_date").isNull() | (c("installation_date") <= F.current_date())),
    ]
)

silver_meter = publish(mtr, mtr_dups, "silver_meter_master", E_MTR,
                       ["meter_id"], MTR_RULES, bronze_mtr.count())

In [0]:
# --------------------------------------------------
# 6. SMART METER READINGS (telemetry, business key reading_id)
# Uniqueness, missingness, type/range validation and business-rule quarantine
# --------------------------------------------------
E_RDG = "smart_meter_readings"
ARRIVAL_ORDER = lambda rcv: [F.col(rcv).desc_nulls_last(), F.col("_ingested_at").desc_nulls_last(),
                             F.col("_source_file").desc_nulls_last()]


def prepare_readings(df, meter_ref, substation_ref):
    """Dedup (reading_id, then meter+interval replay), parent flags, arrival flags."""
    order = ARRIVAL_ORDER("received_ts")
    df, d1 = dedupe_with_audit(df, ["reading_id"], order, E_RDG, "RDG-DUP-001", "DUPLICATE")
    df, d2 = dedupe_with_audit(df, ["meter_id", "interval_start_ts"], order, E_RDG, "RDG-DUP-002", "REPLAY")
    df = add_ref_flag(df, meter_ref, ["meter_id"], "_ref_meter_ok")
    df = add_ref_flag(df, substation_ref, ["substation_id"], "_ref_substation_ok")
    df = df.join(F.broadcast(meter_ref.select("meter_id", F.col("substation_id").alias("_ref_meter_substation"))
                                      .dropDuplicates(["meter_id"])), "meter_id", "left")
    df = flag_arrival(df, ["meter_id"], "interval_end_ts", "received_ts", LATE_READING_MINUTES)
    df = df.withColumn("reading_date", F.to_date("interval_start_ts"))
    return df, d1.unionByName(d2, allowMissingColumns=True)


def readings_rules():
    e, s1, s2 = E_RDG, ENERGY_SENTINELS[0], ENERGY_SENTINELS[1]
    dur = (F.unix_timestamp("interval_end_ts") - F.unix_timestamp("interval_start_ts")) / 60
    return (
        not_null_rules(e, "RDG", ["reading_id", "meter_id", "substation_id",
                                  "interval_start_ts", "interval_end_ts", "energy_kwh"])     # 001-006
        + [
            rescued_rule(e, "RDG-007"),
            rule(e, "RDG-008", "ERROR", ["interval_start_ts", "interval_end_ts"],
                 "Chronology validation failed: interval_end_ts must be after interval_start_ts",
                 c("interval_end_ts") > c("interval_start_ts")),
            rule(e, "RDG-009", "ERROR", ["interval_start_ts", "interval_end_ts"],
                 f"Interval validation failed: interval length must be {INTERVAL_MINUTES} minutes",
                 dur == INTERVAL_MINUTES),
            rule(e, "RDG-010", "ERROR", ["received_ts", "interval_end_ts"],
                 "Chronology validation failed: received_ts before interval_end_ts",
                 c("received_ts").isNull() | (c("received_ts") >= c("interval_end_ts"))),
            rule(e, "RDG-011", "ERROR", ["energy_kwh"],
                 f"Sentinel validation failed: energy_kwh = {s1} (missing-reading placeholder)",
                 c("energy_kwh") != s1),
            rule(e, "RDG-012", "ERROR", ["energy_kwh"],
                 f"Sentinel validation failed: energy_kwh = {s2} (overflow placeholder)",
                 c("energy_kwh") != s2),
            rule(e, "RDG-013", "ERROR", ["energy_kwh"],
                 f"Range validation failed: energy_kwh must be between 0 and {ENERGY_MAX_KWH} per interval",
                 c("energy_kwh").isin(*ENERGY_SENTINELS) | c("energy_kwh").between(0, ENERGY_MAX_KWH)),
            rule(e, "RDG-014", "ERROR", ["voltage_v"],
                 f"Range validation failed: voltage_v must be between {VOLTAGE_RANGE[0]} and {VOLTAGE_RANGE[1]}",
                 c("voltage_v").isNull() | c("voltage_v").between(*VOLTAGE_RANGE)),
            rule(e, "RDG-015", "ERROR", ["current_a"],
                 f"Range validation failed: current_a must be between {CURRENT_RANGE[0]} and {CURRENT_RANGE[1]}",
                 c("current_a").isNull() | c("current_a").between(*CURRENT_RANGE)),
            rule(e, "RDG-016", "ERROR", ["power_factor"],
                 "Range validation failed: power_factor must be between 0 and 1",
                 c("power_factor").isNull() | c("power_factor").between(*POWER_FACTOR_RANGE)),
            rule(e, "RDG-017", "ERROR", ["quality_code"],
                 "Domain validation failed: quality_code must be VALID or ESTIMATED",
                 c("quality_code").isNull() | c("quality_code").isin("VALID", "ESTIMATED")),
            rule(e, "RDG-018", "ERROR", ["meter_id"],
                 "Referential validation failed: meter_id not found in silver_meter_master",
                 c("meter_id").isNull() | c("_ref_meter_ok")),
            rule(e, "RDG-019", "ERROR", ["substation_id"],
                 "Referential validation failed: substation_id not found in silver_substation_master",
                 c("substation_id").isNull() | c("_ref_substation_ok")),
        ]
        + [rule(e, f"RDG-{20 + i:03d}", "WARN", [col_],
                f"Null validation warning: {col_} missing (optional attribute)", c(col_).isNotNull())
           for i, col_ in enumerate(["voltage_v", "current_a", "power_factor",
                                     "quality_code", "reading_source"])]                       # 020-024
        + [
            rule(e, "RDG-025", "WARN", ["substation_id"],
                 "Consistency warning: substation_id differs from meter master",
                 c("_ref_meter_substation").isNull() | (c("substation_id") == c("_ref_meter_substation"))),
            rule(e, "RDG-026", "WARN", ["received_ts"],
                 f"Timeliness warning: received more than {LATE_READING_MINUTES} min after interval_end_ts",
                 ~c("_is_late_arrival")),
            rule(e, "RDG-027", "WARN", ["interval_start_ts", "received_ts"],
                 "Ordering warning: arrived after a later interval of the same meter (out-of-order)",
                 ~c("_is_out_of_order")),
        ]
    )


RDG_RULES = readings_rules()
bronze_rdg = read_bronze("bronze_smart_meter_readings")
rdg, rdg_dups = prepare_readings(bronze_rdg, silver_meter, silver_substation)

silver_readings = publish(rdg, rdg_dups, "silver_smart_meter_readings", E_RDG,
                          ["reading_id"], RDG_RULES, bronze_rdg.count())

In [0]:
# --------------------------------------------------
# 7. GRID OUTAGE EVENTS (event feed, business key outage_id)
# Event timestamp, parent linkage, event-type and chronology checks
# --------------------------------------------------
E_OUT = "grid_outage_events"


def prepare_outages(df, substation_ref):
    df, d = dedupe_with_audit(df, ["outage_id"], ARRIVAL_ORDER("reported_ts"), E_OUT, "OUT-DUP-001")
    df = add_ref_flag(df, substation_ref, ["substation_id"], "_ref_substation_ok")
    df = flag_arrival(df, ["substation_id"], "outage_start_ts", "reported_ts", LATE_OUTAGE_REPORT_MINUTES)
    df = df.withColumn("outage_duration_minutes",
                       (F.unix_timestamp("outage_end_ts") - F.unix_timestamp("outage_start_ts")) / 60)
    return df, d


def outage_rules():
    e = E_OUT
    return (
        not_null_rules(e, "OUT", ["outage_id", "substation_id", "outage_start_ts",
                                  "outage_cause", "incident_status"])                         # 001-005
        + [
            rescued_rule(e, "OUT-006"),
            rule(e, "OUT-007", "ERROR", ["outage_start_ts", "outage_end_ts"],
                 "Chronology validation failed: outage_end_ts before outage_start_ts",
                 c("outage_end_ts").isNull() | (c("outage_end_ts") >= c("outage_start_ts"))),
            rule(e, "OUT-008", "ERROR", ["reported_ts", "outage_start_ts"],
                 "Chronology validation failed: reported_ts before outage_start_ts",
                 c("reported_ts").isNull() | (c("reported_ts") >= c("outage_start_ts"))),
            rule(e, "OUT-009", "ERROR", ["incident_status", "outage_end_ts"],
                 "Status validation failed: RESOLVED outage must have outage_end_ts",
                 (c("incident_status") != "RESOLVED") | c("outage_end_ts").isNotNull()),
            rule(e, "OUT-010", "ERROR", ["outage_cause"],
                 "Event-type validation failed: outage_cause must be PLANNED, EQUIPMENT or STORM",
                 c("outage_cause").isNull() | c("outage_cause").isin("PLANNED", "EQUIPMENT", "STORM")),
            rule(e, "OUT-011", "ERROR", ["severity"],
                 "Domain validation failed: severity must be LOW, MEDIUM, HIGH or CRITICAL",
                 c("severity").isNull() | c("severity").isin("LOW", "MEDIUM", "HIGH", "CRITICAL")),
            rule(e, "OUT-012", "ERROR", ["incident_status"],
                 "Domain validation failed: incident_status must be OPEN, INVESTIGATING or RESOLVED",
                 c("incident_status").isNull() | c("incident_status").isin("OPEN", "INVESTIGATING", "RESOLVED")),
            rule(e, "OUT-013", "ERROR", ["affected_meters"],
                 "Range validation failed: affected_meters must be >= 0",
                 c("affected_meters").isNull() | (c("affected_meters") >= 0)),
            rule(e, "OUT-014", "ERROR", ["substation_id"],
                 "Parent linkage validation failed: substation_id not found in silver_substation_master",
                 c("substation_id").isNull() | c("_ref_substation_ok")),
            rule(e, "OUT-015", "WARN", ["severity"],
                 "Null validation warning: severity missing (optional attribute)", c("severity").isNotNull()),
            rule(e, "OUT-016", "WARN", ["reported_ts"],
                 f"Timeliness warning: reported more than {LATE_OUTAGE_REPORT_MINUTES} min after outage_start_ts",
                 ~c("_is_late_arrival")),
            rule(e, "OUT-017", "WARN", ["outage_start_ts", "reported_ts"],
                 "Ordering warning: reported after a later outage of the same substation (out-of-order)",
                 ~c("_is_out_of_order")),
        ]
    )


OUT_RULES = outage_rules()
bronze_out = read_bronze("bronze_grid_outage_events")
out, out_dups = prepare_outages(bronze_out, silver_substation)

silver_outages = publish(out, out_dups, "silver_grid_outage_events", E_OUT,
                         ["outage_id"], OUT_RULES, bronze_out.count())

In [0]:
# --------------------------------------------------
# 8. METER RECOVERY EVENTS (supplementary feed, business key recovery_event_id)
# Lifecycle ordering, references, replay and status transition checks
# Fixes: signal_strength is dBm (negative is valid); acknowledged_flag (Y/N) was
# not parseable by the Bronze boolean schema and sits in _rescued_data -> recovered.
# --------------------------------------------------
E_REC = "meter_recovery_events"


def prepare_recovery(df, meter_ref, outage_ref):
    rescued_ack = F.get_json_object(F.col("_rescued_data"), "$.acknowledged_flag")
    df = (df.withColumn("_orig_acknowledged_flag", rescued_ack)
            .withColumn("acknowledged_flag",
                        F.coalesce(F.col("acknowledged_flag"), flag_expr(rescued_ack))))
    order = ARRIVAL_ORDER("received_ts")
    df, d1 = dedupe_with_audit(df, ["recovery_event_id"], order, E_REC, "REC-DUP-001", "DUPLICATE")
    df, d2 = dedupe_with_audit(df, ["meter_id", "outage_id", "event_ts"], order, E_REC, "REC-DUP-002", "REPLAY")
    df = add_ref_flag(df, meter_ref, ["meter_id"], "_ref_meter_ok")
    df = add_ref_flag(df, outage_ref, ["outage_id"], "_ref_outage_ok")
    df = df.join(F.broadcast(meter_ref.select("meter_id", F.col("substation_id").alias("_ref_meter_substation"))
                                      .dropDuplicates(["meter_id"])), "meter_id", "left")
    df = df.join(F.broadcast(outage_ref.select(
                    "outage_id",
                    F.col("outage_start_ts").alias("_ref_outage_start"),
                    F.col("outage_end_ts").alias("_ref_outage_end"),
                    F.col("incident_status").alias("_ref_outage_status"),
                    F.col("substation_id").alias("_ref_outage_substation")).dropDuplicates(["outage_id"])),
                 "outage_id", "left")
    df = (df.withColumn("minutes_from_outage_start",
                        (F.unix_timestamp("event_ts") - F.unix_timestamp("_ref_outage_start")) / 60)
            .withColumn("outage_status_at_load", F.col("_ref_outage_status"))
            .withColumn("recovery_seq_for_meter", F.row_number().over(
                Window.partitionBy("meter_id").orderBy("event_ts", "recovery_event_id"))))
    df = flag_arrival(df, ["meter_id"], "event_ts", "received_ts", LATE_RECOVERY_MINUTES)
    return df, d1.unionByName(d2, allowMissingColumns=True)


def recovery_rules():
    e = E_REC
    return (
        not_null_rules(e, "REC", ["recovery_event_id", "meter_id", "outage_id",
                                  "event_ts", "recovery_type"])                               # 001-005
        + [
            rescued_rule(e, "REC-006", handled=("acknowledged_flag",)),
            rule(e, "REC-007", "ERROR", ["received_ts", "event_ts"],
                 "Chronology validation failed: received_ts before event_ts",
                 c("received_ts").isNull() | (c("received_ts") >= c("event_ts"))),
            rule(e, "REC-008", "ERROR", ["recovery_type"],
                 "Event-type validation failed: recovery_type must be AUTO or MANUAL",
                 c("recovery_type").isNull() | c("recovery_type").isin("AUTO", "MANUAL")),
            rule(e, "REC-009", "ERROR", ["signal_strength"],
                 f"Range validation failed: signal_strength must be between {SIGNAL_DBM_RANGE[0]} and {SIGNAL_DBM_RANGE[1]} dBm",
                 c("signal_strength").isNull() | c("signal_strength").between(*SIGNAL_DBM_RANGE)),
            rule(e, "REC-010", "ERROR", ["acknowledged_flag"],
                 "Domain validation failed: acknowledged_flag must be Y/N",
                 c("_orig_acknowledged_flag").isNull() | c("acknowledged_flag").isNotNull()),
            rule(e, "REC-011", "ERROR", ["meter_id"],
                 "Referential validation failed: meter_id not found in silver_meter_master",
                 c("meter_id").isNull() | c("_ref_meter_ok")),
            rule(e, "REC-012", "ERROR", ["outage_id"],
                 "Parent linkage validation failed: outage_id not found in silver_grid_outage_events",
                 c("outage_id").isNull() | c("_ref_outage_ok")),
            rule(e, "REC-013", "WARN", ["event_ts", "outage_id"],
                 "Lifecycle warning: recovery event_ts before linked outage_start_ts",
                 c("_ref_outage_start").isNull() | (c("event_ts") >= c("_ref_outage_start"))),
            rule(e, "REC-014", "WARN", ["outage_id"],
                 "Status transition warning: recovery recorded while linked outage is still OPEN",
                 c("_ref_outage_status").isNull() | (c("_ref_outage_status") != "OPEN")),
            rule(e, "REC-015", "WARN", ["meter_id", "outage_id"],
                 "Consistency warning: meter substation differs from outage substation",
                 c("_ref_meter_substation").isNull() | c("_ref_outage_substation").isNull()
                 | (c("_ref_meter_substation") == c("_ref_outage_substation"))),
            rule(e, "REC-016", "WARN", ["acknowledged_flag"],
                 "Null validation warning: acknowledged_flag missing (optional attribute)",
                 c("acknowledged_flag").isNotNull()),
            rule(e, "REC-017", "WARN", ["received_ts"],
                 f"Timeliness warning: received more than {LATE_RECOVERY_MINUTES} min after event_ts",
                 ~c("_is_late_arrival")),
            rule(e, "REC-018", "WARN", ["event_ts", "received_ts"],
                 "Ordering warning: arrived after a later recovery of the same meter (out-of-order)",
                 ~c("_is_out_of_order")),
        ]
    )


REC_RULES = recovery_rules()
bronze_rec = read_bronze("bronze_meter_recovery_events")
rec, rec_dups = prepare_recovery(bronze_rec, silver_meter, silver_outages)

silver_recovery = publish(rec, rec_dups, "silver_meter_recovery_events", E_REC,
                          ["recovery_event_id"], REC_RULES, bronze_rec.count())

In [0]:
# --------------------------------------------------
# 9. PROJECT BUSINESS LOGIC
# Generate expected meter intervals -> detect gaps -> aggregate energy -> reconcile daily totals
# --------------------------------------------------
SLOT_SEC = INTERVAL_MINUTES * 60
SLOT_INTERVAL = F.expr(f"INTERVAL {INTERVAL_MINUTES} MINUTES")


def to_slot(col):
    """Floor a timestamp to its 5-minute interval slot."""
    return F.timestamp_seconds(F.floor(F.unix_timestamp(col) / SLOT_SEC) * SLOT_SEC)


def build_expected_intervals(readings, window_start, window_end):
    """One row per meter per expected 5-min slot in [window_start, window_end],
    joined to what was actually reported."""
    grid = (readings.select("meter_id").distinct()
                    .withColumn("slot_ts", F.explode(F.sequence(
                        F.lit(window_start).cast("timestamp"),
                        F.lit(window_end).cast("timestamp"), SLOT_INTERVAL))))
    actual = (readings.withColumn("slot_ts", to_slot("interval_start_ts"))
                      .groupBy("meter_id", "slot_ts")
                      .agg(F.count("reading_id").alias("readings_in_slot"),
                           F.sum("energy_kwh").alias("energy_kwh")))
    return (grid.join(actual, ["meter_id", "slot_ts"], "left")
                .withColumn("is_reported", F.col("readings_in_slot").isNotNull())
                .withColumn("reading_date", F.to_date("slot_ts")))


def detect_gaps(expected):
    """Collapse consecutive missing slots per meter into gap ranges (gaps-and-islands)."""
    w = Window.partitionBy("meter_id").orderBy("slot_ts")
    return (expected.filter(~F.col("is_reported"))
                    .withColumn("_grp", (F.unix_timestamp("slot_ts") / SLOT_SEC).cast("long")
                                        - F.row_number().over(w))
                    .groupBy("meter_id", "_grp")
                    .agg(F.min("slot_ts").alias("gap_start_ts"),
                         (F.max("slot_ts") + SLOT_INTERVAL).alias("gap_end_ts"),
                         F.count(F.lit(1)).alias("missing_intervals"))
                    .withColumn("gap_minutes", F.col("missing_intervals") * INTERVAL_MINUTES)
                    .drop("_grp"))


def daily_energy(expected, readings):
    """Meter-day grain: interval completeness + energy aggregates (+ peak load in kW)."""
    slots = (expected.groupBy("meter_id", "reading_date")
                     .agg(F.count(F.lit(1)).alias("expected_intervals"),
                          F.sum(F.col("is_reported").cast("int")).alias("reported_intervals")))
    rd = (readings.groupBy("meter_id", "reading_date")
                  .agg(F.count("reading_id").alias("reading_count"),
                       F.round(F.sum("energy_kwh"), 4).alias("total_kwh"),
                       F.max("energy_kwh").alias("peak_interval_kwh"),
                       F.sum((F.col("quality_code") == "ESTIMATED").cast("int")).alias("estimated_readings"),
                       F.sum(F.col("_is_late_arrival").cast("int")).alias("late_readings"),
                       F.sum(F.col("_is_out_of_order").cast("int")).alias("out_of_order_readings")))
    zero_cols = ["reading_count", "total_kwh", "estimated_readings", "late_readings", "out_of_order_readings"]
    return (slots.join(rd, ["meter_id", "reading_date"], "left").fillna(0, subset=zero_cols)
                 .withColumn("missing_intervals", F.col("expected_intervals") - F.col("reported_intervals"))
                 .withColumn("missing_interval_rate",
                             F.round(F.col("missing_intervals") / F.col("expected_intervals"), 6))
                 .withColumn("peak_load_kw",
                             F.round(F.col("peak_interval_kwh") * (60 / INTERVAL_MINUTES), 4)))


# Observation window = first to last valid interval slot in Silver readings
win = silver_readings.agg(F.min(to_slot("interval_start_ts")).alias("s"),
                          F.max(to_slot("interval_start_ts")).alias("e")).first()
print("Expected-interval window:", win["s"], "->", win["e"])

expected_intervals = build_expected_intervals(silver_readings, win["s"], win["e"])
_overwrite(expected_intervals.withColumn("_silver_run_id", F.lit(run_id)), "silver_meter_expected_intervals")
expected_intervals = spark.table(fq("silver_meter_expected_intervals"))

_overwrite(detect_gaps(expected_intervals).withColumn("_silver_run_id", F.lit(run_id)), "silver_meter_interval_gaps")
_overwrite(daily_energy(expected_intervals, silver_readings).withColumn("_silver_run_id", F.lit(run_id)),
           "silver_meter_daily_energy")
silver_gaps = spark.table(fq("silver_meter_interval_gaps"))
silver_daily = spark.table(fq("silver_meter_daily_energy"))

print("expected intervals:", expected_intervals.count(),
      "| gap ranges:", silver_gaps.count(), "| meter-days:", silver_daily.count())
display(silver_daily.orderBy("reading_date", "meter_id").limit(20))

In [0]:
# --------------------------------------------------
# 10. RECONCILIATION CONTROLS (expected vs actual)
# --------------------------------------------------
recon = []

def check(check_id, name, expected, actual, tol=0.0):
    diff = round(float(actual) - float(expected), 4)
    recon.append((check_id, name, float(expected), float(actual), diff,
                   "PASS" if abs(diff) <= tol else "FAIL"))

# R01-R06: every Bronze row is accounted for (valid + quarantined + duplicates)
for i, m in enumerate(RUN_METRICS.values(), start=1):
    check(f"R{i:02d}", f"{m['entity']}: bronze rows = valid + quarantined + duplicates",
          m["bronze_rows"], m["silver_valid"] + m["quarantined"] + m["duplicates_removed"])

# R07: no duplicate business keys in curated Silver tables
keys = {"silver_substation_master": ["substation_id"], "silver_tariff_lookup": ["tariff_code"],
        "silver_meter_master": ["meter_id"], "silver_smart_meter_readings": ["reading_id"],
        "silver_grid_outage_events": ["outage_id"], "silver_meter_recovery_events": ["recovery_event_id"]}
dup_keys = sum(spark.table(fq(t)).count() - spark.table(fq(t)).select(*k).distinct().count()
               for t, k in keys.items())
check("R07", "Duplicate business keys across curated Silver tables", 0, dup_keys)

# R08: total kWh - independent sum over interval readings vs meter-day aggregate
kwh_readings = silver_readings.agg(F.sum("energy_kwh")).first()[0]
kwh_daily = silver_daily.agg(F.sum("total_kwh")).first()[0]
check("R08", "Total kWh: interval readings vs silver_meter_daily_energy", kwh_readings, kwh_daily, tol=0.05)

# R09: day-level kWh reconciliation (days with |difference| > 0.01 kWh)
by_day_r = silver_readings.groupBy("reading_date").agg(F.sum("energy_kwh").alias("r"))
by_day_d = silver_daily.groupBy("reading_date").agg(F.sum("total_kwh").alias("d"))
bad_days = (by_day_r.join(by_day_d, "reading_date", "full")
                    .filter(F.abs(F.coalesce("r", F.lit(0)) - F.coalesce("d", F.lit(0))) > 0.01).count())
check("R09", "Days where daily kWh total does not reconcile", 0, bad_days)

# R10-R12: interval completeness arithmetic
tot = silver_daily.agg(F.sum("expected_intervals").alias("e"), F.sum("reported_intervals").alias("r"),
                       F.sum("missing_intervals").alias("m"), F.sum("reading_count").alias("n")).first()
check("R10", "Reported intervals = valid Silver readings", silver_readings.count(), tot["r"])
check("R11", "Expected intervals = reported + missing", tot["e"], tot["r"] + tot["m"])
check("R12", "Missing intervals: gap table vs meter-day table", tot["m"],
      silver_gaps.agg(F.sum("missing_intervals")).first()[0] or 0)

recon_df = spark.createDataFrame(
    recon, "check_id string, check_name string, expected double, actual double, difference double, status string"
).withColumn("silver_run_id", F.lit(run_id))
_overwrite(recon_df, "silver_reconciliation")
display(recon_df)

print(f"Missing interval rate (overall): {tot['m'] / tot['e']:.4%}  "
      f"({tot['m']:,} of {tot['e']:,} expected 5-min intervals)")

In [0]:
# --------------------------------------------------
# 11. DQ REGISTER OUTPUTS
# --------------------------------------------------
# Rule catalogue (rule ID, severity, columns, reason, handling action)
rule_catalog_df = spark.createDataFrame(
    list(RULE_CATALOG.values()),
    "entity string, rule_id string, severity string, columns string, reason string, action string")
_overwrite(rule_catalog_df, "silver_dq_rule_catalog")

# Unified rejected-record audit (validation quarantine + dropped duplicates), with source file + raw record
rejected_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                      [f for frames in REJECTED_FRAMES.values() for f in frames])
_overwrite(rejected_all, "silver_rejected_records")

# Duplicate / replay audit
_overwrite(reduce(lambda a, b: a.unionByName(b), DUPLICATE_FRAMES.values()), "silver_duplicate_audit")

# Run log (append - one row per entity per run, for trend monitoring)
run_log = (spark.createDataFrame(list(RUN_METRICS.values()))
                .withColumn("silver_run_id", F.lit(run_id))
                .withColumn("run_ts", F.current_timestamp()))
run_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(fq("silver_dq_run_log"))

# Rule hit summary: errors (quarantine/dedup) and warnings (kept rows)
error_hits = (spark.table(fq("silver_rejected_records"))
                   .select("entity", F.explode("rule_ids").alias("rule_id")).groupBy("entity", "rule_id").count())
warn_hits = reduce(lambda a, b: a.unionByName(b), [
    spark.table(fq(t)).select(F.lit(e).alias("entity"), F.explode("_dq_warning_rule_ids").alias("rule_id"))
    for t, e in [(m["silver_table"], m["entity"]) for m in RUN_METRICS.values()]
]).groupBy("entity", "rule_id").count()

display(rule_catalog_df.join(error_hits.unionByName(warn_hits), ["entity", "rule_id"], "left")
                       .withColumnRenamed("count", "rows_hit").fillna(0, subset=["rows_hit"])
                       .orderBy("entity", "rule_id"))

In [0]:
# --------------------------------------------------
# 12. BEFORE vs AFTER COMPARISON
# "Before" = Silver as it existed when this run started (snapshot cell)
# --------------------------------------------------
after_counts_df = spark.createDataFrame(
    [(m["silver_table"], m["bronze_rows"], m["silver_valid"], m["quarantined"],
      m["duplicates_removed"], m["valid_with_warnings"]) for m in RUN_METRICS.values()],
    "table string, bronze_rows long, valid_after long, quarantined_after long, "
    "duplicates_removed_after long, valid_with_warnings_after long")
print("1) Row counts")
display(pre_counts_df.join(after_counts_df, "table", "full")
        .select("table", "bronze_rows", "valid_before", "valid_after",
                "quarantined_before", "quarantined_after",
                "duplicates_removed_after", "valid_with_warnings_after"))

# Quarantine reasons: before (as stored) vs after (one row per individual reason)
before_r = spark.createDataFrame(pre_reasons or [("-", "-", 0)],
                                 "table string, dq_reason string, rows long").withColumn("state", F.lit("BEFORE"))
after_r = reduce(lambda a, b: a.unionByName(b), [
    spark.table(fq(t.replace("silver_", "quarantine_")))
         .select(F.lit(t).alias("table"), F.explode(F.split("_dq_reason", "; ")).alias("dq_reason"))
    for t in SILVER_TABLES]).groupBy("table", "dq_reason").agg(F.count(F.lit(1)).alias("rows")) \
    .withColumn("state", F.lit("AFTER"))
print("2) Quarantine reasons")
display(before_r.unionByName(after_r).orderBy("table", "state", F.desc("rows")))

# Key business metrics before vs after
r_after = silver_readings.agg(F.round(F.sum("energy_kwh"), 2).alias("kwh"), F.max("energy_kwh").alias("mx"),
                              F.sum((F.col("energy_kwh") > ENERGY_MAX_KWH).cast("int")).alias("ex")).first()
after_metrics = {
    "Total kWh in silver readings": r_after["kwh"],
    "Max interval kWh in silver readings": r_after["mx"],
    f"Silver readings with energy_kwh > {ENERGY_MAX_KWH}": r_after["ex"],
    "Meter tariff_code values with hidden CR/LF":
        silver_meter.filter(F.col("tariff_code").rlike("[\\r\\n]")).count(),
    "Recovery events with acknowledged_flag populated":
        silver_recovery.filter(F.col("acknowledged_flag").isNotNull()).count(),
}
print("3) Business metrics")
display(spark.createDataFrame(
    [(k, str(pre_metrics.get(k)), str(v)) for k, v in after_metrics.items()],
    "metric string, before string, after string"))

In [0]:
# --------------------------------------------------
# 13. UNIT TESTS (pytest-style test_* functions, run in-notebook on small synthetic frames)
# Exercise the same pure functions the pipeline uses - no tables are written.
# --------------------------------------------------
from datetime import datetime as _dt, timedelta as _td

T0 = _dt(2026, 7, 1, 5, 0)
RD_SCHEMA = ("reading_id string, meter_id string, substation_id string, interval_start_ts timestamp, "
             "interval_end_ts timestamp, energy_kwh double, voltage_v double, current_a double, "
             "power_factor double, quality_code string, reading_source string, received_ts timestamp, "
             "_rescued_data string, _source_file string, _ingested_at timestamp")
T_METERS = spark.createDataFrame([("M1", "S1"), ("M2", "S1")], "meter_id string, substation_id string")
T_SUBS = spark.createDataFrame([("S1",)], "substation_id string")


def reading(rid, meter="M1", start_min=0, energy=1.0, lag_min=35, pf=0.9, quality="VALID", src="batch_001"):
    s = T0 + _td(minutes=start_min)
    e = s + _td(minutes=INTERVAL_MINUTES)
    return (rid, meter, "S1", s, e, energy, 230.0, 10.0, pf, quality, "READI_1",
            e + _td(minutes=lag_min), None, src, T0)


def run_readings(rows):
    df = clean_strings(add_lineage(spark.createDataFrame(rows, RD_SCHEMA)))
    prepared, dropped = prepare_readings(df, T_METERS, T_SUBS)
    valid, rejected = split_valid_rejected(evaluate_rules(prepared, readings_rules()))
    return valid, rejected, dropped


def test_valid_reading_passes():
    v, r, d = run_readings([reading("R1")])
    assert (v.count(), r.count(), d.count()) == (1, 0, 0)
    assert v.first()["_dq_warnings"] == []


def test_invalid_readings_quarantined_with_specific_reason():
    v, r, d = run_readings([
        reading("R1", energy=-999.0),
        reading("R2", start_min=5, energy=999999.99),
        reading("R3", start_min=10, pf=1.5),
        reading("R4", start_min=15, quality="BAD"),
        reading("R5", meter="M9", start_min=20),
        reading(None, start_min=25),
    ])
    got = {row["reading_id"]: row["_dq_reason"] for row in r.collect()}
    assert v.count() == 0 and len(got) == 6
    assert got["R1"] == "Sentinel validation failed: energy_kwh = -999.0 (missing-reading placeholder)"
    assert got["R2"] == "Sentinel validation failed: energy_kwh = 999999.99 (overflow placeholder)"
    assert got["R3"] == "Range validation failed: power_factor must be between 0 and 1"
    assert got["R4"] == "Domain validation failed: quality_code must be VALID or ESTIMATED"
    assert got["R5"] == "Referential validation failed: meter_id not found in silver_meter_master"
    assert got[None] == "Null validation failed: reading_id"


def test_exact_duplicate_removed_and_audited():
    v, r, d = run_readings([reading("R1"), reading("R1", src="batch_002")])
    assert (v.count(), r.count()) == (1, 0)
    assert [x["_duplicate_type"] for x in d.collect()] == ["EXACT_DUPLICATE"]
    assert v.first()["_duplicate_count"] == 1


def test_conflicting_duplicate_keeps_latest_received():
    v, r, d = run_readings([reading("R1", energy=1.0, lag_min=35),
                            reading("R1", energy=2.0, lag_min=40)])
    assert v.first()["energy_kwh"] == 2.0
    assert d.first()["_duplicate_type"] == "CONFLICTING_DUPLICATE"


def test_replay_same_meter_interval_new_id_is_dropped():
    v, r, d = run_readings([reading("R1"), reading("R1-REPLAY", lag_min=40)])
    assert v.count() == 1 and d.first()["_duplicate_type"].endswith("_REPLAY")


def test_late_arrival_flagged_but_kept():
    v, r, d = run_readings([reading("R1", lag_min=LATE_READING_MINUTES + 30)])
    row = v.first()
    assert r.count() == 0 and row["_is_late_arrival"] is True
    assert "RDG-026" in row["_dq_warning_rule_ids"]


def test_out_of_order_flagged_but_kept():
    # R2 (interval 10 min) arrives first; R1 (interval 0 min) arrives later -> out-of-order
    v, r, d = run_readings([reading("R2", start_min=10, lag_min=5),
                            reading("R1", start_min=0, lag_min=30)])
    flags = {x["reading_id"]: x["_is_out_of_order"] for x in v.collect()}
    assert flags == {"R1": True, "R2": False}


def test_replay_of_whole_batch_is_idempotent():
    batch = [reading("R1"), reading("R2", start_min=5)]
    v1, _, _ = run_readings(batch)
    v2, _, d2 = run_readings(batch + batch)
    key = lambda df: sorted((x["reading_id"], x["energy_kwh"]) for x in df.collect())
    assert key(v1) == key(v2) and d2.count() == 2


def test_clean_strings_strips_cr_and_blanks():
    df = clean_strings(spark.createDataFrame([("T01\r", "   ")], "tariff_code string, region string"))
    row = df.first()
    assert row["tariff_code"] == "T01" and row["region"] is None


def test_gap_detection_and_daily_completeness():
    rd = spark.createDataFrame(
        [("R1", "M1", T0, 1.0, "VALID", False, False),
         ("R2", "M1", T0 + _td(minutes=5), 2.0, "VALID", False, False),
         ("R3", "M1", T0 + _td(minutes=20), 3.0, "ESTIMATED", False, False)],
        "reading_id string, meter_id string, interval_start_ts timestamp, energy_kwh double, "
        "quality_code string, _is_late_arrival boolean, _is_out_of_order boolean"
    ).withColumn("reading_date", F.to_date("interval_start_ts"))
    exp = build_expected_intervals(rd, T0, T0 + _td(minutes=20))
    gaps = detect_gaps(exp).collect()
    assert exp.count() == 5
    assert [(g["gap_start_ts"], g["gap_end_ts"], g["missing_intervals"]) for g in gaps] == \
           [(T0 + _td(minutes=10), T0 + _td(minutes=20), 2)]
    day = daily_energy(exp, rd).first()
    assert (day["expected_intervals"], day["reported_intervals"], day["missing_intervals"],
            day["total_kwh"], day["estimated_readings"]) == (5, 3, 2, 6.0, 1)


def test_signal_strength_dbm_range():
    sig_rule = [r for r in recovery_rules() if r["rule_id"] == "REC-009"]
    df = spark.createDataFrame([("OK", -70.0), ("BAD", 5.0)], "id string, signal_strength double")
    v, r = split_valid_rejected(evaluate_rules(df, sig_rule))
    assert [x["id"] for x in v.collect()] == ["OK"] and [x["id"] for x in r.collect()] == ["BAD"]


# ---------------- runner ----------------
_results = []
for _name, _fn in sorted((n, f) for n, f in globals().items() if n.startswith("test_") and callable(f)):
    try:
        _fn()
        _results.append((_name, "PASS", ""))
    except Exception as _e:
        _results.append((_name, "FAIL", f"{type(_e).__name__}: {_e}"[:500]))

display(spark.createDataFrame(_results, "test string, status string, detail string"))
_failed = [n for n, s, _ in _results if s == "FAIL"]
assert not _failed, f"Failed tests: {_failed}"
print(f"All {len(_results)} unit tests passed")